In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")
# df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfTimeComplete.pkl")
# df = df.loc[df["Type"]=="DIVIDEND"]

In [ ]:
df = df.loc[df["Day of Week"]=="Wednesday"]

In [ ]:
# df.groupby("Type").mean()[["VIX return","Sentiment return"]]

In [ ]:
df = df[["VIX return","Sentiment return"]]
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Sentiment return"]!=0]
df = df.loc[df["VIX return"]!=0]
df_dates = df
df_dates["Time"] = df_dates.index
df_dates['Time']=df_dates['Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_dates.index = df_dates['Time']

In [ ]:
df_dates

In [ ]:
events_db = []
for date, row in df_dates.T.iteritems():
    new_row = {'event_date': np.datetime64(row["Time"]),'market_ticker':'Sentiment return','security_ticker': 'VIX return'}#row["Company"]}
    events_db.append(new_row)

In [ ]:
returns = df_dates[["Sentiment return","VIX return"]]
returns.index=df_dates.Time

In [ ]:
len(df_dates)

In [ ]:
len(events_db)

In [ ]:
returns.index.names = ['date']

In [ ]:
returns.to_csv("returns_list.csv")

In [ ]:
returns

In [ ]:
es.Single.import_returns("returns_list.csv")
# es.Single.import_FamaFrench("../CSV/FamaFrenchSPY.csv")

In [ ]:
tickers={"Sentiment return"}

In [ ]:
events=events_db

In [ ]:
# tickers = ['SPY']
releases = dict()
agg = es.Multiple.from_list(
        events,
        event_study_model = es.Single.market_model,
        event_window= (-15,+15)
    )

In [ ]:
len(events)

In [ ]:
event = agg
event.plot()
print(event)
# plt.figure(figsize=(20, 20))
plt.xlabel('Event Day')
plt.title('Event Study - Minute Frequency\nSentiment Return and VIX Return')
# plt.show()

In [ ]:
x = range(3, 6)
for n in x:
  print(n)

In [ ]:
abnormalreturns_df = pd.DataFrame(data=agg.AAR,index=(range(-15,16)))

In [ ]:
abnormalreturns_df

In [ ]:
mean_AAR = abnormalreturns_df.mean(axis = 1)
var_AAR = (abnormalreturns_df.std())**2
var_matrix = pd.DataFrame(var_AAR)
var_matrix = var_matrix.T
var_AAR = sum(var_matrix.iloc[0])/16**2
Std_AAR = np.sqrt(var_AAR)
mean_AAR.plot()
plt.rcParams.update({'font.size': 12})
plt.xlabel('Event Window')
plt.ylabel('Average Abnormal Return')
plt.title('Sentiment')
plt.axhline(y=Std_AAR*1.96,color='red',linestyle='--')
plt.axhline(y=Std_AAR*-1.96,color='red',linestyle='--')

In [ ]:
agg.AAR

# PLOT CAR

In [ ]:
abnormalreturns_df = pd.DataFrame(data=agg.get_CAR_dist())

In [ ]:
abnormalreturns_df

In [ ]:
CAR_df = abnormalreturns_df.cumsum()
plt.figure(figsize=(24,16))
for i in range(1,17):
    plt.subplot(4,4,i)
    CAR_df[CAR_df.columns[i-1]].plot()
    # plt.plot(se_cum_single.iloc[:,i-1]*1.96, color='red',linestyle='--')
    # plt.plot(se_cum_single.iloc[:,i-1]*-1.96, color='red',linestyle='--')
    plt.xlabel('Event Window')
    plt.ylabel('CAR')
    plt.title(CAR_df.columns[i-1])

Var_AAR = ((CAR_df.mean(axis=1))**2)/16
Std_AAR = np.sqrt(Var_AAR)
# CAAR
CAAR = mean_AAR.cumsum()
# Plot CAAR
CAAR.plot(figsize=(12,8))
plt.xlabel("Event Window")
plt.plot(se*1.96, color='red',linestyle='--')
plt.plot(se*-1.96, color='red',linestyle='--')
plt.ylabel("Cumulative Return")
plt.title("Cumulative Average Abnormal Return")